In [31]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

pd.set_option( 'display.max_columns' , None )
df = pd.read_csv(r'C:\Users\Samak\OneDrive\เดสก์ท็อป\Fraud Detection\Data\fraud_oracle.csv')
df.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,Age,Fault,PolicyType,VehicleCategory,VehiclePrice,FraudFound_P,PolicyNumber,RepNumber,Deductible,DriverRating,Days_Policy_Accident,Days_Policy_Claim,PastNumberOfClaims,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,21,Policy Holder,Sport - Liability,Sport,more than 69000,0,1,12,300,1,more than 30,more than 30,none,3 years,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,34,Policy Holder,Sport - Collision,Sport,more than 69000,0,2,15,400,4,more than 30,more than 30,none,6 years,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,47,Policy Holder,Sport - Collision,Sport,more than 69000,0,3,7,400,3,more than 30,more than 30,1,7 years,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,65,Third Party,Sedan - Liability,Sport,20000 to 29000,0,4,4,400,2,more than 30,more than 30,1,more than 7,51 to 65,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,27,Third Party,Sport - Collision,Sport,more than 69000,0,5,3,400,1,more than 30,more than 30,none,5 years,31 to 35,No,No,External,none,no change,1 vehicle,1994,Collision


### Feature Engineering

In [ ]:
# Feature 1: Is it a weekend accident/claim?
df['Is_Weekend_Accident'] = df['DayOfWeek'].isin(['Sat', 'Sun']).astype(int)
df['Is_Weekend_Claim'] = df['DayOfWeekClaimed'].isin(['Sat', 'Sun']).astype(int)

# Feature 2: High Risk Driver Profile
# Based on EDA: Age <= 35 and Utility Vehicle
df['HighRisk_Utility_Young'] = ((df['Age'] <= 35) & (df['Age'] > 0) & (df['VehicleCategory'] == 'Utility')).astype(int)
# Based on EDA: Age=0 and Sedan
df['HighRisk_Sedan_NoAge'] = ((df['Age'] == 0) & (df['VehicleCategory'] == 'Sedan')).astype(int)

# Feature 3: Claimed in a different month than the accident
df['Is_Cross_Month_Claim'] = (df['Month'] != df['MonthClaimed']).astype(int)

# Feature 4: Holiday Season Accident (Dec, Jan)
df['Is_HolidaySeason_Accident'] = df['Month'].isin(['Dec', 'Jan']).astype(int)

# Evaluate these features
print("Feature: Is_Weekend_Accident")
print(df.groupby('Is_Weekend_Accident')['FraudFound_P'].mean())

print("\nFeature: HighRisk_Utility_Young")
print(df.groupby('HighRisk_Utility_Young')['FraudFound_P'].mean())

print("\nFeature: HighRisk_Sedan_NoAge")
print(df.groupby('HighRisk_Sedan_NoAge')['FraudFound_P'].mean())

print("\nFeature: Is_CrossMonth_Claim")
print(df.groupby('Is_Cross_Month_Claim')['FraudFound_P'].mean())



Feature: Is_Weekend_Accident
Is_Weekend_Accident
0    0.059857
Name: FraudFound_P, dtype: float64

Feature: HighRisk_Utility_Young
HighRisk_Utility_Young
0    0.059280
1    0.188406
Name: FraudFound_P, dtype: float64

Feature: HighRisk_Sedan_NoAge
HighRisk_Sedan_NoAge
0    0.058654
1    0.173913
Name: FraudFound_P, dtype: float64

Feature: Is_CrossMonth_Claim
Is_Cross_Month_Claim
0    0.055322
1    0.072901
Name: FraudFound_P, dtype: float64


In [33]:
print(df.shape)

(15420, 39)


### 4 Features ใหม่ที่สร้างขึ้นมาเพื่อจับพฤติกรรมมิจฉาชีพโดยเฉพาะ:

#### HighRisk_Utility_Young (วัยรุ่น/วัยทำงานตอนต้น ที่ขับรถอเนกประสงค์)

- แนวคิด: จากที่เราวิเคราะห์ไปว่าคนอายุ 16-35 ปีที่ขับ Utility Car มีอัตราการเกิด Fraud สูงทะลุเพดาน

- การสร้าง: Age <= 35 และขับรถ Utility

- ผลลัพธ์ที่ได้: กลุ่มที่เข้าเงื่อนไขนี้มี Fraud Rate พุ่งกระฉูดถึง 18.84% (เทียบกับเคสทั่วไปที่อยู่แค่ ~5.9%) Feature นี้จะเป็นตัวเตือน (Red Flag) ชั้นดีให้โมเดลเลยครับ!

#### HighRisk_Sedan_NoAge (ขับรถเก๋งแต่ตั้งใจปกปิดอายุ)

- แนวคิด: คนที่ Age = 0 และขับ Sedan มักจะมีปัญหาอะไรบางอย่างแอบแฝง

- การสร้าง: Age == 0 และขับ Sedan

- ผลลัพธ์ที่ได้: กลุ่มนี้มีอัตราทุจริตสูงถึง 17.39% โมเดลจะสามารถเรียนรู้พฤติกรรมการจงใจไม่กรอกข้อมูลในกลุ่มรถยนต์ยอดฮิตได้

#### Is_CrossMonth_Claim (อุบัติเหตุเดือนนึง เคลมอีกเดือนนึง)

- แนวคิด: การแจ้งเคลมข้ามเดือน อาจหมายถึงความล่าช้าในการเตรียมการจัดฉาก หรือพยายามหาจังหวะที่เหมาะสม

- การสร้าง: เทียบว่า Month (เดือนเกิดเหตุ) ไม่ตรงกับ MonthClaimed (เดือนที่แจ้งเคลม)

- ผลลัพธ์ที่ได้: เคสที่แจ้งเคลมข้ามเดือนมี Fraud Rate 7.29% (สูงกว่าเคสที่แจ้งเคลมเดือนเดียวกันซึ่งอยู่ที่ 5.53%)

#### Is_Weekend_Accident (อุบัติเหตุช่วงวันหยุดสุดสัปดาห์)

- แนวคิด: มิจฉาชีพอาจเลือกลงมือในวันเสาร์-อาทิตย์ เพราะคิดว่าตรวจสอบยาก หรือบริษัทประกันหยุดทำการ

- การสร้าง: DayOfWeek เป็น 'Saturday' หรือ 'Sunday'

- ผลลัพธ์ที่ได้: อุบัติเหตุวันหยุดมีโอกาสเป็น Fraud 6.81% (เทียบกับวันธรรมดาที่ 5.72%)

## List
#### ตัดคอลัมน์ที่ไม่จำเป็นทิ้ง (Drop Unnecessary Columns)

- ตัดคอลัมน์ PolicyNumber และ RepNumber ทิ้ง เพราะมันเป็นเพียง ID หรือหมายเลขประจำตัวที่ไม่มีผลต่อการเกิดทุจริต (Fraud) หากเก็บไว้โมเดลอาจจะไปจำตัวเลขเหล่านี้แทนที่จะเรียนรู้รูปแบบที่แท้จริง

#### จัดการปัญหา "อายุ = 0" (Handling Missing Age)

- อย่างที่เราพบว่า Age = 0 คือ Red Flag สำคัญ ผมจึงสร้างคอลัมน์ใหม่ชื่อ Age_is_0 (ถ้าใช่เป็น 1 - ถ้าไม่ใช่เป็น 0) เพื่อให้โมเดลรับรู้ว่าเคสนี้มีการปกปิดหรือไม่มีข้อมูลอายุ

- จากนั้น แทนที่ค่า 0 ในคอลัมน์ Age ด้วย ค่ามัธยฐาน (Median) ของคนที่อายุมากกว่า 0 (ซึ่งคืออายุ 38 ปี) เพื่อให้ข้อมูลอายุสามารถนำไปคำนวณทางสถิติต่อได้โดยไม่ผิดเพี้ยน

#### แปลงข้อมูลกลุ่มที่มีลำดับ (Ordinal Encoding)

- ข้อมูลหลายตัวเป็นข้อความที่มี "ลำดับความมาก-น้อย" แฝงอยู่ ผมจึงแปลงข้อความเป็นตัวเลขตามลำดับ เช่น:

- PastNumberOfClaims (ประวัติการเคลม): 'none' = 0, '1' = 1, '2 to 4' = 2, 'more than 4' = 3

- VehiclePrice (ราคารถ): 'less than 20000' = 0 ไล่ไปจนถึง 'more than 69000' = 5

- แปลงเดือนและวันเป็นตัวเลข (Jan = 1, Monday = 1 เป็นต้น)

- กลุ่มอายุของกรมธรรม์, วันที่เกิดเหตุจนถึงวันเคลม ฯลฯ ถูกแปลงเป็นตัวเลขเพื่อบ่งบอกระยะเวลาทั้งหมด

#### แปลงข้อมูลกลุ่มที่ไม่มีลำดับ (One-Hot Encoding)

- ข้อมูลอย่าง ยี่ห้อรถ (Make), พื้นที่เกิดเหตุ (AccidentArea), เพศ (Sex), ฝ่ายผิด (Fault), ประเภทรถ (VehicleCategory) เหล่านี้ไม่สามารถให้ค่าเป็น 1,2,3 ได้ เพราะ Honda ไม่ได้มีค่ามากกว่า Toyota

- ผมใช้เทคนิค One-Hot Encoding แตกคอลัมน์เหล่านี้ออกเป็นหลายๆ คอลัมน์ (เช่น คอลัมน์ Make_Honda ถ้าใช่จะเป็น True ถ้าไม่ใช่จะเป็น False)

- ข้อควรระวังที่ทำไปแล้ว: ผมใช้คำสั่ง drop_first=True เพื่อลบคอลัมน์แรกสุดของแต่ละกลุ่มทิ้ง (เช่น ลบ Male เหลือแค่ Female) เพื่อป้องกันปัญหา Multicollinearity (คอลัมน์ซ้ำซ้อนกันจนกวนโมเดล)

In [34]:
# Drop unnecessary columns
df.drop(['PolicyNumber', 'RepNumber'], axis=1, inplace=True)


#Change 0 to null values in 'Age' column to impute later
df['Age'] = df['Age'].replace(0, np.nan)

# Re-create 'Age_Range' based on imputed age (or we can just keep Age)
# Let's keep it numerical for the models to learn from, 
# or we can ordinally encode the age ranges.

# Ordinal Encoding for categorical variables that have a natural order
ordinal_cols = {
    'Month': {'Jan':1, 'Feb':2, 'Mar':3, 'Apr':4, 'May':5, 'Jun':6, 
              'Jul':7, 'Aug':8, 'Sep':9, 'Oct':10, 'Nov':11, 'Dec':12},
    'MonthClaimed': {'Jan':1, 'Feb':2, 'Mar':3, 'Apr':4, 'May':5, 'Jun':6, 
                     'Jul':7, 'Aug':8, 'Sep':9, 'Oct':10, 'Nov':11, 'Dec':12, '0':0}, # Notice '0' in MonthClaimed
    'DayOfWeek': {'Monday':1, 'Tuesday':2, 'Wednesday':3, 'Thursday':4, 'Friday':5, 'Saturday':6, 'Sunday':7},
    'DayOfWeekClaimed': {'Monday':1, 'Tuesday':2, 'Wednesday':3, 'Thursday':4, 'Friday':5, 'Saturday':6, 'Sunday':7, '0':0},
    'PastNumberOfClaims': {'none':0, '1':1, '2 to 4':2, 'more than 4':3},
    'NumberOfSuppliments': {'none':0, '1 to 2':1, '3 to 5':2, 'more than 5':3},
    'VehiclePrice': {'less than 20000':0, '20000 to 29000':1, '30000 to 39000':2, 
                     '40000 to 59000':3, '60000 to 69000':4, 'more than 69000':5},
    'AgeOfVehicle': {'new':0, '2 years':1, '3 years':2, '4 years':3, '5 years':4, '6 years':5, '7 years':6, 'more than 7':7},
    'Days_Policy_Accident': {'none':0, '1 to 7':1, '8 to 15':2, '15 to 30':3, 'more than 30':4},
    'Days_Policy_Claim': {'none':0, '8 to 15':1, '15 to 30':2, 'more than 30':3},
    'AddressChange_Claim': {'no change':0, 'under 6 months':1, '1 year':2, '2 to 3 years':3, '4 to 8 years':4},
    'NumberOfCars': {'1 vehicle':1, '2 vehicles':2, '3 to 4':3, '5 to 8':4, 'more than 8':5}
}

for col, mapping in ordinal_cols.items():
    df[col] = df[col].map(mapping)

# Use LabelEncoder for age of policy holder since it's grouped text
le = LabelEncoder()
df['AgeOfPolicyHolder'] = le.fit_transform(df['AgeOfPolicyHolder'])

# One-Hot Encoding for nominal (unordered) categorical variables
nominal_cols = ['Make', 'AccidentArea', 'Sex', 'MaritalStatus', 'Fault', 'PolicyType', 
                'VehicleCategory', 'PoliceReportFiled', 'WitnessPresent', 'AgentType', 'BasePolicy']
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True) # drop_first to avoid multicollinearity

print("Data shape after Feature Engineering:", df.shape)
print("\nSample Data (first 2 rows):")
print(df.head(2))

Data shape after Feature Engineering: (15420, 65)

Sample Data (first 2 rows):
   Month  WeekOfMonth  DayOfWeek  DayOfWeekClaimed  MonthClaimed  \
0     12            5          3                 2             1   
1      1            3          3                 1             1   

   WeekOfMonthClaimed   Age  VehiclePrice  FraudFound_P  Deductible  \
0                   1  21.0             5             0         300   
1                   4  34.0             5             0         400   

   DriverRating  Days_Policy_Accident  Days_Policy_Claim  PastNumberOfClaims  \
0             1                     4                  3                   0   
1             4                     4                  3                   0   

   AgeOfVehicle  AgeOfPolicyHolder  NumberOfSuppliments  AddressChange_Claim  \
0             2                  3                    0                    2   
1             5                  4                    0                    0   

   NumberOfCars  Yea

In [35]:
# Reverting back to check if there are any remaining nulls after mapping
null_counts = df.isnull().sum()
if null_counts.sum() > 0:
    print("Nulls found after mapping:\n", null_counts[null_counts > 0])
else:
    print("No nulls found. Encoding successful.")

Nulls found after mapping:
 Age    320
dtype: int64


In [36]:
# Save processed dataframe to CSV
df.to_csv('fraud_oracle_fe.csv', index=False, encoding='utf-8')
print("Saved fraud_oracle_fe.csv")

Saved fraud_oracle_fe.csv
